# MECH 309: Assignment 4, Question 2

_Cagri Arslan_

February 19, 2025

*All work can be found on https://github.com/imported-canuck/MECH-309*

In [1]:
# Imports
import numpy as np
from utils import *

### a)

Orthogonal iteration is used to find all eigenpairs of a matrix simultaneously. It then re-orthogonalizes at each step to prevent iterates from collapsing onto the same direction. 

The "re-orthogonalization" step requires a QR factorization. Hence, a QR factorization helper is produced first. Our formulation of QR factorization uses a Gram-Schmidt orthogonalization algorithm.

In [ ]:
def QR_factorization(A):
    '''
    Factorizes a matrix A into an orthogonal matrix Q and an upper triangular
    matrix R using the Gram-Schmidt process.

    Inputs:
        A: m x n matrix to be factorized

    Returns:
        Q: m x n orthogonal matrix
        R: n x n upper triangular matrix
    '''
    m = A.shape[0] # number of rows of A
    n = A.shape[1] # number of columns of A

    Q = np.zeros((m, n)) # initialize Q
    R = np.zeros((n, n)) # initialize R

    for k in range(n):
        q_k = A[:, k:k+1] # get the i-th column of A

        for j in range(k): # iterate over the previous columns of Q
            q_j = Q[:, j:j+1] # get the j-th column of Q
            r_jk = q_j.T @ q_k # compute the (j, k)-th element of R
            R[j, k] = r_jk.item() # assign the (j, k)-th element of R
            # subtract the projection of q_k onto q_j from q_k
            q_k = q_k - r_jk * q_j 
        
        # compute the (k, k)-th diagonal element of R as the 2-norm of q_k
        r_kk = np.linalg.norm(q_k, ord = 2)

        # if r_kk is zero, it means that the columns of A are linearly 
        # dependent and we cannot proceed with the factorization
        if r_kk == 0:
            raise ValueError("The columns of A are linearly dependent.")
        
        q_k = q_k / r_kk # normalize the k-th column of Q

        Q[:, k:k+1] = q_k # assign the k-th column of Q
        R[k, k] = r_kk # assign the k-th diagonal element of R

    return Q, R


Now that we have a script that performs QR factorization, we can use it to perform orthogonal iteration on a square matrix $A$.

In [ ]:
def orthogonal_iteration(A, max_iters = 1000, tol = 1e-5):
    '''
    Computes the eigenvalues and eigenvectors of a matrix A using orthogonal
    iteration.

    Inputs:
        A: n x n matrix

    Returns:
        eigenvalues: n x 1 vector of eigenvalues
        eigenvectors: n x n matrix of eigenvectors 
        iters: number of iterations taken to converge
    '''
    n = A.shape[0] # number of rows of A
    counter = 0 # iteration counter

    if A.shape[1] != n:
        raise ValueError("Input matrix A must be square.")

    # Initialize X0 as a random n x n matrix
    X = np.random.rand(n, n)

    # Orthonormalize initial guess (Q0)
    Q, _ = QR_factorization(X)

    while counter < max_iters:
        # Form X_k = A Q_{k-1}
        X = A @ Q

        # QR factorization of X_k  -> Q_k, R_k
        Q_new, R = QR_factorization(X)

        # Check for convergence (Q has stabilized)
        if np.linalg.norm(Q_new - Q, ord='fro') < tol:
            Q = Q_new
            break

        Q = Q_new
        counter += 1

    # Failed to converge, return result but print warning
    if counter == max_iters:
        print("Warning: Orthogonal iteration did not converge within the maximum number of iterations.")
        
    # Eigenvalue estimates from Rayleigh quotients
    T = Q.T @ A @ Q                   # eigenvectors already normalized
    eigenvalues = np.diag(T)          # shape (n,)
    eigenvectors = Q                  # columns are eigenvector estimates

    # Sort eigenvalues and eigenvectors by descending abs value of eigenvalues
    idx = np.argsort(-np.abs(eigenvalues.reshape(-1)))
    eigenvalues = eigenvalues.reshape(-1)[idx].reshape(n,1)
    eigenvectors = eigenvectors[:, idx]

    return eigenvalues, eigenvectors, counter

In [8]:
# --- load A from matrix.csv ---
A = np.loadtxt("matrix.csv", delimiter=",")

# --- run orthogonal iteration ---
eigvals, eigvecs, iters = orthogonal_iteration(A, max_iters=5000, tol=1e-8)

# --- print akk eigenvalues (no eigenvector matrix) ---
eigvals = np.asarray(eigvals).reshape(-1)      # (n,)
idx = np.argsort(-np.abs(eigvals))             # sort by |lambda| desc
eigvals_sorted = eigvals[idx]

print("Orthogonal Iteration Eigenvalues")
print("-" * 36)
print(f"A shape: {A.shape}")
print(f"Iterations to convergence: {iters}")
print(f"Count: {eigvals_sorted.size}\n")

for i, lam in enumerate(eigvals_sorted, start=1):
    print(f"{i:3d})  lambda = {lam: .12e}")

Orthogonal Iteration Eigenvalues
------------------------------------
A shape: (10, 10)
Iterations to convergence: 631
Count: 10

  1)  lambda =  6.636336061769e-01
  2)  lambda =  6.023876495200e-01
  3)  lambda =  5.575189334180e-01
  4)  lambda =  4.816439016310e-01
  5)  lambda =  4.227629767949e-01
  6)  lambda =  3.743913825727e-01
  7)  lambda =  3.657157623523e-01
  8)  lambda =  2.498468184972e-01
  9)  lambda =  1.755380817218e-01
 10)  lambda =  9.999999731518e-02


### b)

Matrix $A$ **is positive definite**. All of its eigenvalues are positive. 

### c)

We calculate the condition number by dividing the largest eigenvalue by the smallest eigenvalue. As eigenvalues are already sorted in descending order, we simply divide the "first" listed eigenvalue by the "last" listed eigenvalue. For a symmetric (more generally, normal) matrix:

$$
\textrm{cond}_{2}(A) = \frac{\textrm{max}\lambda}{\textrm{min}\lambda}
$$

More generally, the condition number for a nonsingular square matrix (irrespect of structure) is:

$$
\textrm{cond}_{2}(A) = ||A^{-1}||_{2}||A||_{2}
$$

Both methods yield near-identical condition numbers of $\kappa \approx 6.636$

In [9]:
n, m = A.shape
if n != m:
    raise ValueError("A must be square.")

# --- calculated: eigenvalue ratio (symmetric A => real evals) ---
evals = np.linalg.eigvalsh(A)          # ascending
cond_calc = evals[-1] / evals[0]       # lambda_max / lambda_min

# --- true: definition using norms and inverse ---
cond_true = np.linalg.norm(A, 2) * np.linalg.norm(np.linalg.inv(A), 2)

# --- residual ---
residual = abs(cond_calc - cond_true) / abs(cond_true)

print("Condition number (2-norm)")
print("-" * 26)
print(f"Calculated (λ_max/λ_min):   {cond_calc: .12e}")
print(f"True (||A||2||A^-1||2):     {cond_true: .12e}")
print(f"Residual:                   {residual: .3e}")

Condition number (2-norm)
--------------------------
Calculated (λ_max/λ_min):    6.636336239943e+00
True (||A||2||A^-1||2):      6.636336239943e+00
Residual:                    2.677e-16
